In [1]:
param = {
    "test": "power",
    "sample_size": 200,
    "batch_size": 128,
    "z_dim": 1,
    "dx": 1,
    "dy": 1,
    "n_test": 100,
    "epochs_num": 100,
    "eps_std": 1.0,
    "dist_z": 'gaussian',
    "alpha_x": 0.20,
    "m_value": 100,
    "k_value": 8,
    "j_value": 1000,
    "noise_dimension": 5,
    "noise_dimension_type": "normal",
    "noise_dimension_var": 1,
    "hidden_layer_size": 1024,
    "normal_ini": False,
    "preprocess": 'scale',
    "G_lr": 7e-5,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 0,
    "using_orcale": False,
    "lambda_1": 1,
    "lambda_2": 1,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-4,
    "lambda_4": 2e-5,
    "drop_out_p": 0.2,
    "is_sparse": True,
    "sparse_ratio": 0.05,
    "lambda_quantile": 0.5,
    "quantile_samples": 20
}

import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm

# [修改1] 删除全局 device，改为在 mGAN() 内部按 gpu_id 动态创建


def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0, alpha_x=0.05,
                             preprocess="None", dist_z='gaussian'):
    num = size
    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), 1 * torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), 1 * torch.eye(dy))
    if dist_z == 'gaussian':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))
    elif dist_z == 'laplace':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))
    m = TD.Bernoulli(torch.tensor([alpha_x]))
    Axy = torch.ones((dx, dy)) * alpha_x
    if sType == 'H0':
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        X = Z + nstd * error_x
        Y = Z + nstd * error_y
    else:
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        delta = m.sample((num,))
        X = Z + (1 - delta) * nstd * error_x + delta * nstd * error_y
        Y = Z + nstd * error_y
    return X, Y, Z


def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0,
                                          alpha_x=0.05, normalize=True, seed=None, dist_z='gaussian'):
    num = size
    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), 1 * torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), 1 * torch.eye(dy))
    Axy = torch.ones((dx, dy)) * alpha_x
    m = TD.Bernoulli(torch.tensor([alpha_x]))
    if sType == 'H0':
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        X = Z + nstd * error_x
        Y = Z + nstd * error_y
    else:
        error_x = error_generator_x.sample((num,))
        error_y = error_generator_y.sample((num,))
        delta = m.sample((num,))
        X = Z + (1 - delta) * nstd * error_x + delta * nstd * error_y
        Y = Z + nstd * error_y
    return X, Y


def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch, x_torch, y_torch, z_torch,
                        sigma_w, sigma_u=1, sigma_v=1, boor_rv_type="gaussian",
                        device=None):  # [修改2] 新增 device 参数
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    w_mx = torch.exp(-w_mx / sigma_w)
    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u),
        dim=2)
    u_mx_3 = u_mx_2.T
    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v),
        dim=2)
    v_mx_3 = v_mx_2.T
    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
        sum_mx = sum_mx + temp_add_mx
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
        sum2_mx = sum2_mx + temp2_add_mx
    u_mx_4 = 1 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4
    v_mx_4 = 1 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4
    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n).to(device))  # [修改2]
    stat = 1 / (n - 1) * torch.sum(FF_mx).item()
    boottemp = np.array([])
    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num)).to(device)  # [修改2]
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num).to(device)              # [修改2]
    for bb in range(boot_num):
        random_mx = torch.matmul(eboot[:, bb].reshape(-1, 1), eboot[:, bb].reshape(-1, 1).T)
        bootmatrix = FF_mx * random_mx
        stat_boot = 1 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)
    return stat, boottemp


class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index], \
               self.Z_real[(self.batch_size + index) % self.sample_size]


class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
    def __init__(self, Y, Z, batch_size):
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = Z.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.Y_real[index], self.Z_real[index]


def sample_noise(sample_size, noise_dimension, noise_type, input_var=1,
                 device=None):  # [修改3] 新增 device 参数
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),           # [修改3]
            input_var * torch.eye(noise_dimension).to(device)) # [修改3]
        Z = noise_generator.sample((sample_size,))
    if noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension)
    if noise_type == "Cauchy":
        Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample(
            (sample_size, noise_dimension)).squeeze(2)
    return Z


class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension, hidden_layer_size,
                 BN_type, ReLU_coef, drop_out_p, drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
        self.sigmoid = torch.nn.Sigmoid()
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
        self.drop_input = drop_input

    def forward(self, x):
        if self.BN_type:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
            x = self.fc_last(x)
        else:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
            x = self.fc_last(x)
        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        # [修改3] 不在构造函数中 .to(device)，由外层统一 .to(device)
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias)
        self.mask = functools.reduce(
            torch.block_diag, [torch.ones(size_out, size_in) for i in range(m)])

    def forward(self, x):
        # [修改3] mask 动态跟随模型所在设备
        self.linear.weight.data *= self.mask.to(self.linear.weight.device)
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension, hidden_layer_size,
                 BN_type, ReLU_coef, hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0])
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for i in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)
        return self.linear8(output)


def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1, n) - Y.repeat(1, n).T) / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    mx_1_2 = torch.exp(-mx_1_2 / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-torch.abs(Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-torch.abs(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1, n) - Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=2, dim=2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2) / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-(Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def pinball_loss(y_true, y_pred, tau):
    err = y_true - y_pred
    return torch.mean(torch.max(tau * err, (tau - 1.0) * err))


def train_ver3(
        G_zx, G_zy,
        X, Y, Z, X_test, Y_test, Z_test, M,
        noise_dimension, noise_type, G_lr, hidden_layer_size,
        DataLoader, BN_type, ReLU_coef,
        lambda_quantile=0.5, quantile_samples=20,
        epochs_num=50,
        patience=5, min_delta=1e-5,
        sigma_z=1, sigma_x=1, sigma_y=1,
        normal_ini=False,
        lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
        lambda_3=0, drop_out_p=0.2, noise_dimension_var=1,
        lambda_4=0,
        device=None):  # [修改3] 新增 device 参数

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改3]
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改3]
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改3]
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改3]

        if normal_ini:
            for p in G_zy.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改3]
            for p in G_zx.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改3]

    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    iter_count = 0
    G_zy = G_zy.train()
    G_zx = G_zx.train()

    best_loss = float('inf')
    counter = 0

    for epoch in range(epochs_num):
        batch_count = 0
        G_zy = G_zy.train()
        G_zx = G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)   # [修改3]
            Y_real = Y_real.to(device)   # [修改3]
            Z_real = Z_real.to(device)   # [修改3]
            Z_fake = Z_fake.to(device)   # [修改3]

            batch_size = Z_real.shape[0]
            Z_repeated = Z_real.repeat_interleave(quantile_samples, dim=0)

            Noise_for_quantile = sample_noise(
                Z_repeated.shape[0], noise_dimension, noise_type,
                input_var=noise_dimension_var, device=device).to(device)  # [修改3]

            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)  # [修改3]
            Y_fake = G_zy(torch.cat((Z_real, Noise_fake), dim=1)).to(device)
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)  # [修改3]
            X_fake = G_zx(torch.cat((Z_real, Noise_fake), dim=1)).to(device)

            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_quantile), dim=1))
            Y_generated_reshaped = Y_generated_group.reshape(batch_size, quantile_samples, -1)
            q_25_y = torch.quantile(Y_generated_reshaped, q=0.25, dim=1)
            q_50_y = torch.quantile(Y_generated_reshaped, q=0.50, dim=1)
            q_75_y = torch.quantile(Y_generated_reshaped, q=0.75, dim=1)
            loss_q25_y = pinball_loss(Y_real, q_25_y, 0.25)
            loss_q50_y = pinball_loss(Y_real, q_50_y, 0.50)
            loss_q75_y = pinball_loss(Y_real, q_75_y, 0.75)
            loss_quantile_y = 0.15 * loss_q25_y + 0.70 * loss_q50_y + 0.15 * loss_q75_y

            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_quantile), dim=1))
            X_generated_reshaped = X_generated_group.reshape(batch_size, quantile_samples, -1)
            q_25_x = torch.quantile(X_generated_reshaped, q=0.25, dim=1)
            q_50_x = torch.quantile(X_generated_reshaped, q=0.50, dim=1)
            q_75_x = torch.quantile(X_generated_reshaped, q=0.75, dim=1)
            loss_q25_x = pinball_loss(X_real, q_25_x, 0.25)
            loss_q50_x = pinball_loss(X_real, q_50_x, 0.50)
            loss_q75_x = pinball_loss(X_real, q_75_x, 0.75)
            loss_quantile_x = 0.15 * loss_q25_x + 0.70 * loss_q50_x + 0.15 * loss_q75_x

            g_zy_error = None
            G_zy_solver.zero_grad()
            g_zx_error = None
            G_zx_solver.zero_grad()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param_g in G_zy.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param_g, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param_g, ord=1)

            mmd_loss_y = (lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zy_error = mmd_loss_y + lambda_quantile * loss_quantile_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param_g in G_zx.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param_g, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param_g, ord=1)

            mmd_loss_x = (lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zx_error = mmd_loss_x + lambda_quantile * loss_quantile_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

            iter_count += 1
            batch_count += 1

        current_loss = g_zx_error + g_zy_error
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    return G_zy, G_zx


def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_orcale=False, lambda_1=1, lambda_2=1,
         lambda_quantile=0.5, quantile_samples=20,
         using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2,
         noise_dimension_var=1, noise_dimension_type="normal", lambda_4=1,
         gpu_id=0):  # [修改4] 新增 gpu_id 参数

    # [修改4] 在函数内部根据 gpu_id 创建局部 device
    enable_cuda = True
    if torch.cuda.is_available() and enable_cuda:
        device = torch.device(f'cuda:{gpu_id}')
    else:
        device = torch.device('cpu')

    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H0', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H1', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x.to(device), sim_y.to(device), sim_z.to(device)

    w_mx = torch.linalg.vector_norm(
        z.repeat(n, 1, 1) - torch.swapaxes(z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    sigma_w = torch.median(w_mx).item()
    u_mx = torch.exp(-torch.abs(y.repeat(1, n) - y.repeat(1, n).T))
    sigma_u = torch.median(u_mx).item()
    v_mx = torch.exp(-torch.abs(x.repeat(1, n) - x.repeat(1, n).T))
    sigma_v = torch.median(v_mx).item()

    test_size = int(n / k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    if not using_orcale:
        G_zy = Generator(z_dim, y_dims, noise_dimension, hidden_layer_size, False, 0.1, drop_out_p).to(device)
        G_zx = Generator(z_dim, x_dims, noise_dimension, hidden_layer_size, False, 0.1, drop_out_p).to(device)

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))
        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]
        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))
        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)

        if not using_orcale:
            current_epochs = epochs_num if k_fold == 0 else max(10, epochs_num // 5)
            G_zy, G_zx = train_ver3(
                G_zx=G_zx, G_zy=G_zy,
                X=X_train, Y=Y_train, Z=Z_train, M=M,
                X_test=X_test, Y_test=Y_test, Z_test=Z_test,
                noise_dimension=noise_dimension, noise_type=noise_dimension_type,
                G_lr=G_lr, hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
                lambda_quantile=lambda_quantile, quantile_samples=quantile_samples,
                epochs_num=epochs_num,
                sigma_z=sigma_w, sigma_x=sigma_v, sigma_y=sigma_u,
                normal_ini=normal_ini, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, wgt_decay=wgt_decay, lambda_3=lambda_3,
                drop_out_p=drop_out_p, noise_dimension_var=noise_dimension_var,
                lambda_4=lambda_4,
                device=device)  # [修改4] 传入局部 device

        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M, device=device)   # [修改4]
        gen_y_all = torch.zeros(test_size, M, device=device)   # [修改4]
        z_all = torch.zeros(test_size, z_dim, device=device)   # [修改4]
        x_all = torch.zeros(test_size, x_dims, device=device)  # [修改4]
        y_all = torch.zeros(test_size, y_dims, device=device)  # [修改4]
        cur_itr = 0

        if not using_orcale:
            G_zx = G_zx.eval()
            G_zy = G_zy.eval()

        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
            z_test_temp = z_test.repeat(M, 1)
            if not using_orcale:
                z_test_temp = z_test_temp.to(device)                                      # [修改4]
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)              # [修改4]
                with torch.no_grad():
                    fake_x = G_zx(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)              # [修改4]
                with torch.no_grad():
                    fake_y = G_zy(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
            elif using_orcale:
                if simulation == 'type1error':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H0', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)
                elif simulation == 'power':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H1', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)

            gen_x_all[cur_itr, :] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr, :] = fake_y.detach().reshape(-1)
            x_all[cur_itr, :] = x_test
            y_all[cur_itr, :] = y_test
            z_all[cur_itr, :] = z_test
            cur_itr = cur_itr + 1

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all.to(device), gen_y_all.to(device),
            x_all.to(device), y_all.to(device), z_all.to(device),
            sigma_w, sigma_u, sigma_v, boor_rv_type,
            device=device)  # [修改4] 传入局部 device
        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp)
        cur_k = cur_k + 1

        if using_orcale:
            gen_x_50 = torch.quantile(gen_x_all, 0.50, dim=1).reshape(-1, 1)
            gen_y_50 = torch.quantile(gen_y_all, 0.50, dim=1).reshape(-1, 1)
            mse_x = torch.mean((gen_x_50 - x_all) ** 2).item()
            mse_y = torch.mean((gen_y_50 - y_all) ** 2).item()
            print(f'Test MSE x (median diff) [{mse_x}], MSE y (median diff) [{mse_y}]')

    return np.mean(torch.mean(boot_temp_all, dim=0).numpy() > torch.mean(stat_all).item())


# ============================================================
# 并行部分
# ============================================================
from joblib import Parallel, delayed
import multiprocessing

def run_experiment(params):
    test                 = params["test"]
    sample_size          = params["sample_size"]
    batch_size           = params["batch_size"]
    z_dim                = params["z_dim"]
    dx                   = params["dx"]
    dy                   = params["dy"]
    n_test               = params["n_test"]
    epochs_num           = params["epochs_num"]
    eps_std              = params["eps_std"]
    dist_z               = params["dist_z"]
    alpha_x              = params["alpha_x"]
    m_value              = params["m_value"]
    k_value              = params["k_value"]
    j_value              = params["j_value"]
    noise_dimension      = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var  = params["noise_dimension_var"]
    hidden_layer_size    = params["hidden_layer_size"]
    normal_ini           = params["normal_ini"]
    preprocess           = params["preprocess"]
    G_lr                 = params["G_lr"]
    alpha                = params["alpha"]
    alpha1               = params["alpha1"]
    set_seeds            = params["set_seeds"]
    using_orcale         = params["using_orcale"]
    lambda_1             = params["lambda_1"]
    lambda_2             = params["lambda_2"]
    using_Gen            = params["using_Gen"]
    boor_rv_type         = params["boor_rv_type"]
    wgt_decay            = params["wgt_decay"]
    lambda_3             = params["lambda_3"]
    lambda_4             = params["lambda_4"]
    drop_out_p           = params["drop_out_p"]
    lambda_quantile      = params["lambda_quantile"]
    quantile_samples     = params["quantile_samples"]

    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    # [修改4] 获取可用 GPU 数量，控制并发数，轮询分配 GPU
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    num_cores = min(20, n_test)
    if num_cores < 1:
        num_cores = 1

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, "
          f"多重分位数对齐系数: {lambda_quantile}, 实验次数: {n_test}, "
          f"并行核数: {num_cores}, 可用GPU数: {num_gpus}")
    if test == 'power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")

    # [修改4] 按 job_index % num_gpus 轮询分配 GPU
    p_values = Parallel(n_jobs=num_cores)(
        delayed(mGAN)(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2,
            lambda_quantile=lambda_quantile, quantile_samples=quantile_samples,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var,
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4,
            gpu_id=job_index % num_gpus  # [修改4] 轮询分配 GPU
        )
        for job_index, _ in enumerate(range(n_test))  # [修改4] enumerate 获取 job_index
    )

    p_values = np.array(p_values)
    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)

    print(f"\n" + "=" * 50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("=" * 50 + "\n")

    return p_values

In [8]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 100
param["lambda_quantile"] = 0.2
param["quantile_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[14:56:53] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.2, 实验次数: 100, 并行核数: 12, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1100 (at significance level 0.1)
Emp Rej Rate: 0.0500 (at significance level 0.05)

[15:04:08] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.2, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1900 (at significance level 0.0971)
Emp Rej Rate: 0.1100 (at significance level 0.054150000000000004)

[15:09:35] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.2, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5000 (at significance level 0.0971)
Emp Rej Rate: 0.3900 (at significance level 0.054150000000000004)

[15:13:59] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.2, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5800 (at significance level 0.0971)
Emp R

In [2]:
param["sample_size"] = 1000
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 1000
param["lambda_quantile"] = 0.3
param["quantile_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[18:44:58] 开始并行实验...
模式: type1error, 样本量: 1000, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 1000, 并行核数: 20, 可用GPU数: 4


/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.0980 (at significance level 0.1)
Emp Rej Rate: 0.0480 (at significance level 0.05)

[19:29:56] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 1000, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3690 (at significance level 0.103)
Emp Rej Rate: 0.2530 (at significance level 0.05480000000000001)

[20:14:38] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 1000, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.7890 (at significance level 0.103)
Emp Rej Rate: 0.6960 (at significance level 0.05480000000000001)

[20:59:13] 开始并行实验...
模式: power, 样本量: 1000, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 1000, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.9540 (at significance level 0.103)
Emp Rej Rate: 0.9180 (at significance level 0.05480000000000001)

[21:43:46] 开始并行实验...
模式: power, 样本量: 1000, 

In [10]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 100
param["lambda_quantile"] = 0.3
param["quantile_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[15:31:19] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1000 (at significance level 0.1)
Emp Rej Rate: 0.0600 (at significance level 0.05)

[15:35:04] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2300 (at significance level 0.11310000000000002)
Emp Rej Rate: 0.1100 (at significance level 0.04355)

[15:38:43] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3600 (at significance level 0.11310000000000002)
Emp Rej Rate: 0.2300 (at significance level 0.04355)

[15:43:00] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5400 (at significance level 0.11310000000

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 39.50 GiB of which 17.12 MiB is free. Process 607701 has 25.72 GiB memory in use. Process 1152454 has 770.00 MiB memory in use. Process 1291669 has 896.00 MiB memory in use. Process 1292012 has 4.43 GiB memory in use. Process 1296811 has 896.00 MiB memory in use. Including non-PyTorch memory, this process has 632.00 MiB memory in use. Process 1297199 has 896.00 MiB memory in use. Process 1297329 has 896.00 MiB memory in use. Process 1297354 has 896.00 MiB memory in use. Process 1297468 has 896.00 MiB memory in use. Process 1297532 has 896.00 MiB memory in use. Process 1297624 has 896.00 MiB memory in use. Process 1297693 has 896.00 MiB memory in use. Of the allocated memory 82.73 MiB is allocated by PyTorch, and 13.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  S

In [12]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 100
param["lambda_quantile"] = 0.3
param["quantile_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[15:56:15] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1100 (at significance level 0.1)
Emp Rej Rate: 0.0200 (at significance level 0.05)

[16:00:10] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.12

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.7000 (at significance level 0.0959)
Emp Rej Rate: 0.6500 (at significance level 0.06985000000000001)

[16:03:34] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.8900 (at significance level 0.0959)
Emp Rej Rate: 0.8800 (at significance level 0.06985000000000001)

[16:06:56] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.3, 实验次数: 100, 并行核数: 12, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.2

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 1.0000 (at significance level 0.0959)
Emp Rej

In [16]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 100
param["lambda_quantile"] = 0.5
param["quantile_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[19:02:52] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 20, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1200 (at significance level 0.1)
Emp Rej Rate: 0.0600 (at significance level 0.05)

[19:06:18] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2200 (at significance level 0.095)
Emp Rej Rate: 0.1500 (at significance level 0.0453)

[19:09:26] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.08

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3300 (at significance level 0.095)
Emp Rej Rate: 0.2300 (at significance level 0.0453)

[19:12:34] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 20, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5700 (at significance level 0.095)
Emp Rej Rate: 0.4200 (at significanc

In [ ]:
param["test"] = "type1error"

k_value = 2

for sample_size in [400]:
    param["k_value"] = k_value
    param["n_test"]=100
    param["sample_size"] = sample_size
    param["lambda_quantile"] = 0.5
    param["quantile_samples"] = 20
    p_val_list = run_experiment(param)

[11:17:12] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 6

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.0900 (at significance level 0.1)
Emp Rej Rate: 0.0300 (at significance level 0.05)



In [3]:
param["test"] = "power"
param["sample_size"] = 400
param["k_value"] = 2

param["alpha"] = np.quantile(p_val_list, 0.10)
param["alpha1"] = np.quantile(p_val_list, 0.05)

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list = run_experiment(param)

[11:44:34] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 6
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1800 (at significance level 0.109)
Emp Rej Rate: 0.1500 (at significance level 0.0728)

[12:11:39] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 6
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5000 (at significance level 0.109)
Emp Rej Rate: 0.3900 (at significance level 0.0728)

[12:37:28] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 6
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.9300 (at significance level 0.109)
Emp Rej Rate: 0.9100 (at significance level 0.0728)

[13:02:16] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 多重分位数对齐系数: 0.5, 实验次数: 100, 并行核数: 6
备择假设H_1下的模型参数 alpha_x: 0.2

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 1.0000 (at significance level 0.109)
Emp Rej Rate: 1.0000 (at significance level 0.0728)

[13:27:32]